In [1]:
import os
import sys
import requests
import warnings
from PIL import Image

# 1. 忽略警告
warnings.filterwarnings("ignore")

# 2. 【关键】开启 Hugging Face 国内镜像加速（解决下载慢、卡住的问题）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 3. 绕过安全检查
import transformers.utils.import_utils
transformers.utils.import_utils.check_torch_load_is_safe = lambda: None

from transformers import pipeline

In [2]:
MODEL_ID = "8kkillian/autotrain-weather-classification-3723199087"
TEST_IMAGE_URL = "https://images.unsplash.com/photo-1519692933481-e162a57d6721?w=600"
LOCAL_IMAGE_PATH = "test_weather.jpg"

# 下载测试图片
print("1. 正在下载测试图片...")
response = requests.get(TEST_IMAGE_URL, timeout=10)
with open(LOCAL_IMAGE_PATH, "wb") as f:
    f.write(response.content)
print("   图片下载完成！\n")

1. 正在下载测试图片...
   图片下载完成！



In [3]:
print(f"2. 正在通过镜像源下载/加载模型 [{MODEL_ID}] ...")
print("   (如果是第一次运行，下方会出现下载进度条)")

try:
    classifier = pipeline("image-classification", model=MODEL_ID)
    
    image = Image.open(LOCAL_IMAGE_PATH)
    print("\n3. 正在预测天气...")
    predictions = classifier(image)
    
    print("\n" + "="*35)
    print("        天气分类预测结果        ")
    print("="*35)
    for idx, pred in enumerate(predictions, 1):
        print(f" Top {idx}: {pred['label']:<15} 置信度: {pred['score']*100:.2f}%")
    print("="*35)

except Exception as e:
    print(f"\n运行出错: {e}")

2. 正在通过镜像源下载/加载模型 [8kkillian/autotrain-weather-classification-3723199087] ...
   (如果是第一次运行，下方会出现下载进度条)

3. 正在预测天气...

        天气分类预测结果        
 Top 1: rainbow         置信度: 51.87%
 Top 2: snow            置信度: 51.42%
 Top 3: sandstorm       置信度: 49.97%
 Top 4: rain            置信度: 49.58%
 Top 5: lightning       置信度: 46.63%


In [1]:
import os
import sys
import time
import math
import requests
import warnings
import torch
from PIL import Image

# 1. 忽略警告
warnings.filterwarnings("ignore")

# 2. 开启 Hugging Face 国内镜像加速
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 3. 绕过安全检查
import transformers.utils.import_utils
transformers.utils.import_utils.check_torch_load_is_safe = lambda: None

from transformers import pipeline

In [2]:
MODEL_ID = "8kkillian/autotrain-weather-classification-3723199087"
TEST_IMAGE_URL = "https://images.unsplash.com/photo-1519692933481-e162a57d6721?w=600"
LOCAL_IMAGE_PATH = "test_weather.jpg"

In [3]:
print("1. 正在下载测试图片...")
response = requests.get(TEST_IMAGE_URL, timeout=10)
with open(LOCAL_IMAGE_PATH, "wb") as f:
    f.write(response.content)
print("   图片下载完成！\n")

1. 正在下载测试图片...
   图片下载完成！



In [4]:
print(f"2. 正在通过镜像源下载/加载模型 [{MODEL_ID}] ...")

try:
    classifier = pipeline("image-classification", model=MODEL_ID)
    image = Image.open(LOCAL_IMAGE_PATH)
    
    print("\n3. 正在预测天气...")
    
    # ------------------ 新增指标计算准备 ------------------
    # 清空 PyTorch 缓存，以便准确计算显存占用
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    
    # 记录开始时间
    start_time = time.perf_counter()
    
    # 执行推理
    predictions = classifier(image)
    
    # 记录结束时间并计算耗时
    end_time = time.perf_counter()
    latency_ms = (end_time - start_time) * 1000  # 毫秒
    
    # 计算预测分布的香农熵 (Shannon Entropy) - 衡量不确定性
    probs = [pred['score'] for pred in predictions]
    # 防止 log(0)
    entropy = -sum(p * math.log2(p) for p in probs if p > 0)
    
    # 获取 GPU 显存峰值占用 (MB)
    vram_mb = 0
    if torch.cuda.is_available():
        vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
    # ----------------------------------------------------

    print("\n" + "="*40)
    print("          天气分类预测结果          ")
    print("="*40)
    for idx, pred in enumerate(predictions, 1):
        print(f" Top {idx}: {pred['label']:<15} 置信度: {pred['score']*100:.2f}%")
    
    print("="*40)
    print("          附加性能评估指标          ")
    print("="*40)
    print(f" ⏱️ 单张推理耗时 (Latency):    {latency_ms:.2f} ms")
    print(f" 🎲 预测熵 (Prediction Entropy): {entropy:.4f} bits (越高越犹豫)")
    if torch.cuda.is_available():
        print(f" 💾 GPU 显存峰值 (Peak VRAM):   {vram_mb:.2f} MB")
    print("="*40)

except Exception as e:
    print(f"\n运行出错: {e}")

2. 正在通过镜像源下载/加载模型 [8kkillian/autotrain-weather-classification-3723199087] ...

3. 正在预测天气...

          天气分类预测结果          
 Top 1: rainbow         置信度: 51.87%
 Top 2: snow            置信度: 51.42%
 Top 3: sandstorm       置信度: 49.97%
 Top 4: rain            置信度: 49.58%
 Top 5: lightning       置信度: 46.63%
          附加性能评估指标          
 ⏱️ 单张推理耗时 (Latency):    2898.82 ms
 🎲 预测熵 (Prediction Entropy): 2.4999 bits (越高越犹豫)
